# Round analysis scheduler

Run this notebook **during acquisition** in its own JupyterLab tab or kernel,  
alongside `analysis_01_fov_scheduler.ipynb`.

Once all FOVs in a round are processed by the FOV scheduler, this notebook:
- Reads `flip_vertical` from the HAL config XML to set mosaic orientation automatically
- Finds the middle-z slice from the frame table for each color
- Creates one mosaic PNG per color channel (e.g. `round_001_560nm_mosaic.png`)
- Optionally transfers the round data to a network destination during the fluidics window

**Timing logic**

```
   Imaging        Fluidics (~60–100 min)
═══════════════╪═══════════════════════════════════════╪═══ ...
               ▲         ▲                 ▲           ▲
            last file   t_min            t_max      next round
```

* `t_min = 300 s` (5 min) — wait for HAL to finish writing the last files  
* `t_max = 6000 s` (100 min) for **adaptor** fluidics; `3000 s` (50 min) for **direct** readouts

## 1 — Setup

In [ ]:
import os
import sys
import logging
from pathlib import Path

MERCI_DIR  = Path(os.getcwd()).parent          # MERci/ (notebook in MERci/notebooks/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root, e.g. LT048_sample_18/
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config   import ExperimentConfig
from MERci.common.metadata import ExperimentMetadata
from MERci.progress        import ProgressTracker
from MERci.state           import ExperimentStateMonitor
from MERci.scheduler       import FOVScheduler, RoundScheduler
from MERci.visualization   import display_mosaic

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)

print(f"SAMPLE_DIR : {SAMPLE_DIR}")

## 2 — Experiment parameters

Edit the cells below to match your experiment.

In [ ]:
SAMPLE_NAME = SAMPLE_DIR.name

# ── Fluidics type ──────────────────────────────────────────────────────
# "adaptor"  →  t_max = 100 min  (adaptor-based fluidics)
# "direct"   →  t_max =  50 min  (direct-readout fluidics)
FLUIDICS_TYPE = "adaptor"

# ── Image file format (must match what HAL writes) ─────────────────────
IMAGE_SUFFIX = ".zarr"   # options: ".zarr", ".dax", ".tiff"

# ── FOV subset (optional) ──────────────────────────────────────────────
# Set to a list of FOV ids to process only a subset, e.g. [0, 1, 2, 3]
# Leave as None to process all FOVs
FOV_SUBSET = None

# ── Thumbnail frames (optional) ────────────────────────────────────────
# Set to a list of frame indices to limit thumbnail creation, e.g. [2, 3, 4]
# Leave as None to thumbnail all frames
THUMBNAIL_FRAMES = None

# ── Data transfer (optional) ───────────────────────────────────────────
# Set to a network path to copy completed round data during the fluidics
# window.  Transfer starts only when ≥ TRANSFER_MIN_TIME seconds remain.
# Leave as None to skip transfer.
TRANSFER_DEST     = None   # e.g. r"\\NAS\experiments\LT027"
TRANSFER_MIN_TIME = 600    # seconds (10 min)

print(f"Sample name    : {SAMPLE_NAME}")
print(f"Fluidics type  : {FLUIDICS_TYPE}")
print(f"Image suffix   : {IMAGE_SUFFIX}")
print(f"FOV subset     : {FOV_SUBSET}")
print(f"Thumb frames   : {THUMBNAIL_FRAMES}")
print(f"Transfer dest  : {TRANSFER_DEST}")

In [ ]:
config = ExperimentConfig(
    data_dir         = SAMPLE_DIR / "data",
    metadata_dir     = SAMPLE_DIR / "metadata",
    analysis_dir     = SAMPLE_DIR / "analysis",
    settings_dir     = SAMPLE_DIR / "settings",
    round_info_csv   = SAMPLE_DIR / "metadata" / "round_info.csv",
    positions_txt    = SAMPLE_DIR / "positions" / f"positions_{SAMPLE_NAME}.txt",
    fluidics_type    = FLUIDICS_TYPE,
    image_suffix     = IMAGE_SUFFIX,
    fov_subset       = FOV_SUBSET,
    thumbnail_frames = THUMBNAIL_FRAMES,
    transfer_dest    = TRANSFER_DEST,
    transfer_min_time = TRANSFER_MIN_TIME,
)

meta    = ExperimentMetadata.load(config.round_info_csv, config.positions_txt, config.data_dir,
                                   image_suffix=config.image_suffix)
tracker = ProgressTracker(config.analysis_dir)
monitor = ExperimentStateMonitor(config)

print(f"Rounds         : {meta.n_rounds}")
print(f"FOVs           : {meta.n_fovs}")
print(f"t_min / t_max  : {config.t_min:.0f} s / {config.t_max:.0f} s  "
      f"({config.t_min/60:.0f} min / {config.t_max/60:.0f} min)")
print(f"Analysis dir   : {config.analysis_dir}")
if config.transfer_dest:
    print(f"Transfer dest  : {config.transfer_dest}  (min time: {config.transfer_min_time:.0f} s)")

## 3 — Check current progress

Run this cell at any time to see how many FOVs and rounds have been processed.

In [ ]:
summary = tracker.summary(meta)
print(f"FOVs  done : {summary['files_fov_done']} / {summary['files_total']}")
print(f"Rounds done: {summary['rounds_done']} / {summary['rounds_total']}")

## 4 — Round scheduler

**Run this cell and leave it running.**

Once all FOVs in a round are done, it:
- Reads `flip_vertical` from the HAL config XML to set the mosaic orientation automatically
- Finds the middle-z slice from the frame table
- Creates one mosaic PNG per color channel (e.g. `round_001_560nm_mosaic.png`)
- **Transfers the round data** to `TRANSFER_DEST` if set and enough time remains in the fluidics window

Interrupt the kernel (`■` button) to stop the loop gracefully.

In [ ]:
def show_round_phase(phase):
    from IPython.display import clear_output
    clear_output(wait=True)
    tsi = f"{phase.time_since_imaging:.0f} s" if phase.time_since_imaging is not None else "n/a"
    print(f"Phase         : {phase.phase_str}")
    print(f"Time since img: {tsi}")
    print(f"Analyzing     : {phase.should_analyze}")
    smry = tracker.summary(meta)
    print(f"Rounds done   : {smry['rounds_done']} / {smry['rounds_total']}")

RoundScheduler(config, meta, tracker, monitor).run_loop(on_phase_update=show_round_phase)